In [ ]:
#@title 🚀 Ultimate ComfyUI VTON Studio (Final Production Build - Patched)
#@markdown เลือก Model ที่ต้องการติดตั้งลงใน ComfyUI
#@markdown ⚠️ หากต้องการเปลี่ยนโมเดล ให้ Factory Reset Runtime เครื่องก่อนเสมอ
TARGET_NODE = "IDM-VTON" #@param ["CatVTON", "IDM-VTON", "OOTDiffusion", "FitDiT"]

import os
import subprocess
import sys
import threading
import time
import socket

# 🛡️ SYSTEM HACK: ป้องกัน VRAM Fragmentation
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

def run_cmd(cmd, check=False):
    print(f"\n⚙️ Executing: {cmd}")
    process = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in process.stdout:
        sys.stdout.write(line)
    process.wait()
    if check and process.returncode != 0:
        print(f"❌ Command failed: {cmd}")

print("🚀 [1/5] Preparing ComfyUI Core Environment...")
if not os.path.exists("/content/ComfyUI"):
    run_cmd("git clone https://github.com/comfyanonymous/ComfyUI.git /content/ComfyUI")
if not os.path.exists("/content/ComfyUI/custom_nodes/ComfyUI-Manager"):
    run_cmd("git clone https://github.com/ltdrdata/ComfyUI-Manager.git /content/ComfyUI/custom_nodes/ComfyUI-Manager")

os.chdir("/content/ComfyUI")
# 📦 ลง Dependencies หลักของ ComfyUI (แก้บั๊ก ModuleNotFoundError comfy_aimdo)
print("📦 [2/5] Installing ComfyUI Core Requirements & Global Vaccines...")
run_cmd("pip install -q -r requirements.txt")
run_cmd("pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121")
run_cmd("pip install -q aria2")
# ☢️ GLOBAL VACCINE: กัน Numpy 2.0 Apocalypse พังระบบ (ชุบชีวิต onnxruntime)
run_cmd('pip install -q "numpy<2" onnxruntime kornia')

print(f"📦 [3/5] Installing Custom Node for: {TARGET_NODE}...")
os.chdir("/content/ComfyUI/custom_nodes")

if TARGET_NODE == "IDM-VTON":
    # โหลด Node ที่จำเป็นทั้งหมดล่วงหน้า (ไม่ต้องไปกด Manager ให้เสี่ยงพัง)
    required_nodes = [
        "https://github.com/TemryL/ComfyUI-IDM-VTON.git",
        "https://github.com/Fannovel16/comfyui_controlnet_aux.git",
        "https://github.com/storyicon/comfyui_segment_anything.git"
    ]
    for url in required_nodes:
        repo_name = url.split("/")[-1].replace(".git", "")
        if not os.path.exists(repo_name):
            run_cmd(f"git clone {url}")

    print("🔨 [4/5] THE HAMMER FIX (IDM-VTON): Forcing stable dependencies...")
    # ล็อกเวอร์ชันเป๊ะๆ สำหรับ IDM-VTON และลงสมอง (segment-anything) ให้กล่อง SAM
    run_cmd("pip install -q diffusers==0.27.2 huggingface-hub==0.23.2 transformers==4.40.2 accelerate==0.30.0 xformers segment-anything")
    run_cmd("pip install -q -r ComfyUI-IDM-VTON/requirements.txt")
    run_cmd("pip install -q -r comfyui_controlnet_aux/requirements.txt")
    run_cmd("pip install -q -r comfyui_segment_anything/requirements.txt")

elif TARGET_NODE == "CatVTON":
    run_cmd("git clone https://github.com/chflame163/ComfyUI_CatVTON_Wrapper.git 2>/dev/null || true")
    os.chdir("ComfyUI_CatVTON_Wrapper")
    run_cmd("pip install -q -r requirements.txt")
    os.makedirs("../../models/checkpoints", exist_ok=True)
    if not os.path.exists("../../models/checkpoints/catvton.safetensors"):
        run_cmd("aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/zhy6769/CatVTON/resolve/main/catvton.safetensors -d ../../models/checkpoints -o catvton.safetensors")
    print("🔨 [4/5] THE HAMMER FIX (CatVTON): Forcing stable dependencies...")
    run_cmd("pip install -q huggingface-hub==0.23.2 diffusers==0.25.0 transformers==4.38.2")

elif TARGET_NODE == "OOTDiffusion":
    run_cmd("git clone https://github.com/AuroBit/ComfyUI-OOTDiffusion.git 2>/dev/null || true")
    os.chdir("ComfyUI-OOTDiffusion")
    run_cmd("pip install -q -r requirements.txt")
    os.makedirs("checkpoints", exist_ok=True)
    if not os.path.exists("checkpoints/human_parsing.bin"):
        run_cmd("aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/levihsu/OOTDiffusion/resolve/main/checkpoints/human_parsing.bin -d checkpoints -o human_parsing.bin")
    print("🔨 [4/5] THE HAMMER FIX (OOTDiffusion): Forcing stable dependencies...")
    run_cmd("pip install -q huggingface-hub==0.23.2 diffusers==0.25.0 transformers==4.38.2")

elif TARGET_NODE == "FitDiT":
    run_cmd("git clone https://github.com/BoyuanJiang/FitDiT-ComfyUI.git 2>/dev/null || true")
    os.chdir("FitDiT-ComfyUI")
    run_cmd("pip install -q -r requirements.txt")
    print("🔨 [4/5] THE HAMMER FIX (FitDiT): Forcing stable dependencies...")
    run_cmd("pip install -q huggingface-hub==0.23.2 diffusers==0.25.0 transformers==4.38.2")

print("🌐 [5/5] Launching Cloudflare Tunnel...")
os.chdir("/content")
run_cmd("wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb")
run_cmd("dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1")

def iframe_thread(port):
    while True:
        time.sleep(0.5)
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        if sock.connect_ex(('127.0.0.1', port)) == 0: break
        sock.close()

    p = subprocess.Popen(["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{port}"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    for line in p.stderr:
        l = line.decode()
        if "trycloudflare.com " in l:
            print("\n" + "🌟"*25)
            print(f"🚀 COMFYUI READY! คลิกลิงก์ด้านล่างเพื่อเข้าเว็บ 👉:")
            print(f"🔗 URL: {l[l.find('http'):].strip()}")
            print("🌟"*25 + "\n")

threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()

print("🔥 Starting ComfyUI Server...")
os.chdir("/content/ComfyUI")
# 🎯 รันด้วย Granular Precision (แก้บั๊ก ambiguous option --fp16)
!python main.py --dont-print-server --highvram --fp16-unet --fp16-vae

# Any2Anytryon

# ==========================================
# CELL 1: 🛡️ Security, Setup & Dependencies
# ==========================================
import os
from google.colab import userdata
from huggingface_hub import login

print("🔐 [1/3] Authenticating with Hugging Face...")
try:
    hf_token = userdata.get('hf_any2anytryon_use')
    login(token=hf_token)
    print("✅ Auth Success! ได้รับสิทธิ์เข้าถึง FLUX.1-dev แล้ว")
except Exception as e:
    print(f"❌ ERROR: โปรดตั้งค่า Secret Key ชื่อ 'hf_any2anytryon_use' ก่อน!\n{e}")
    raise SystemExit("หยุดการทำงาน")

print("\n⚙️ [2/3] Downloading Repository...")
if not os.path.exists('/content/Any2anyTryon'):
    !git clone https://github.com/logn-2024/Any2anyTryon.git
%cd /content/Any2anyTryon

print("\n📦 [3/3] Installing Dependencies (รอประมาณ 3-5 นาที)...")
!pip install -r requirements.txt
!pip install gradio diffusers accelerate
print("\n✅ Cell 1 เสร็จสมบูรณ์! ลุย Cell 2 ต่อได้เลย")

In [ ]:
#@title 🚀 Ultimate ComfyUI VTON Studio (Colab Edition - Patched v4)
#@markdown เลือก Model ที่ต้องการติดตั้งลงใน ComfyUI
TARGET_NODE = "CatVTON" #@param ["CatVTON", "IDM-VTON", "OOTDiffusion", "FitDiT"]

import os
import subprocess
import sys
import threading
import time
import socket

# ==========================================
# 🔑 COLAB SECRETS & AUTHENTICATION
# ==========================================
print("🔐 [0/5] Checking Colab Secrets for Hugging Face Token...")
try:
    from google.colab import userdata
    from huggingface_hub import login
    hf_token = userdata.get("hf_any2anytryon_use")
    login(token=hf_token)
    print("✅ Auth Success! Logged in to Hugging Face.")
except ImportError:
    print("⚠️ Not running on Colab, skipping secret check.")
except Exception as e:
    print("⚠️ No Secret Key found (or not needed for this public model). Continuing...")

# 🛡️ SYSTEM HACK: ป้องกัน VRAM Fragmentation
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
# เปลี่ยน Path หลักให้เป็นของ Colab
BASE_DIR = "/content"
COMFY_DIR = f"{BASE_DIR}/ComfyUI"

def run_cmd(cmd, check=False):
    print(f"\n⚙️ Executing: {cmd}")
    process = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in process.stdout:
        sys.stdout.write(line)
    process.wait()
    if check and process.returncode != 0:
        print(f"❌ Command failed: {cmd}")

print("\n🚀 [1/5] Preparing ComfyUI Core Environment...")
run_cmd("apt-get update -y && apt-get install -y aria2 git-lfs") 

if not os.path.exists(COMFY_DIR):
    run_cmd(f"git clone https://github.com/comfyanonymous/ComfyUI.git {COMFY_DIR}")
if not os.path.exists(f"{COMFY_DIR}/custom_nodes/ComfyUI-Manager"):
    run_cmd(f"git clone https://github.com/ltdrdata/ComfyUI-Manager.git {COMFY_DIR}/custom_nodes/ComfyUI-Manager")

# 📦 ลง Dependencies หลักของ ComfyUI 
print("📦 [2/5] Installing ComfyUI Core Requirements & Global Vaccines...")
run_cmd(f"pip install -q -r {COMFY_DIR}/requirements.txt")
run_cmd("pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121")
run_cmd("pip install -q huggingface_hub")
# ☢️ GLOBAL VACCINE: กัน Numpy 2.0 Apocalypse
run_cmd('pip install -q "numpy<2" onnxruntime kornia')

print(f"📦 [3/5] Installing Custom Node for: {TARGET_NODE}...")

if TARGET_NODE == "IDM-VTON":
    required_nodes = [
        "https://github.com/TemryL/ComfyUI-IDM-VTON.git",
        "https://github.com/Fannovel16/comfyui_controlnet_aux.git",
        "https://github.com/storyicon/comfyui_segment_anything.git"
    ]
    os.chdir(f"{COMFY_DIR}/custom_nodes")
    for url in required_nodes:
        repo_name = url.split("/")[-1].replace(".git", "")
        if not os.path.exists(repo_name):
            run_cmd(f"git clone {url}")

    print("🧠 [Downloading Brain] กำลังโหลด Weights 10GB+ ของแท้ด้วย HF API (รอ 3-5 นาที)...")
    target_model_dir = f"{COMFY_DIR}/models/IDM-VTON"
    if os.path.exists(target_model_dir) and not os.path.exists(f"{target_model_dir}/unet/diffusion_pytorch_model.bin"):
        run_cmd(f"rm -rf {target_model_dir}")
        
    os.makedirs(target_model_dir, exist_ok=True)
    if not os.path.exists(f"{target_model_dir}/unet/diffusion_pytorch_model.bin"):
        run_cmd(f"huggingface-cli download yisol/IDM-VTON --local-dir {target_model_dir} --local-dir-use-symlinks False --exclude '*.ckpt' '*.pt'")
    
    print("🔨 [4/5] THE HAMMER FIX (IDM-VTON): Forcing stable dependencies...")
    run_cmd("pip install -q diffusers==0.27.2 huggingface-hub==0.23.2 transformers==4.40.2 accelerate==0.30.0 xformers segment-anything")
    run_cmd(f"pip install -q -r {COMFY_DIR}/custom_nodes/ComfyUI-IDM-VTON/requirements.txt")
    run_cmd(f"pip install -q -r {COMFY_DIR}/custom_nodes/comfyui_controlnet_aux/requirements.txt")
    run_cmd(f"pip install -q -r {COMFY_DIR}/custom_nodes/comfyui_segment_anything/requirements.txt")

elif TARGET_NODE == "CatVTON":
    print("🌟 [CatVTON] Installing Main Wrapper & LayerStyle...")
    run_cmd(f"git clone https://github.com/chflame163/ComfyUI_CatVTON_Wrapper.git {COMFY_DIR}/custom_nodes/ComfyUI_CatVTON_Wrapper 2>/dev/null || true")
    run_cmd(f"pip install -q -r {COMFY_DIR}/custom_nodes/ComfyUI_CatVTON_Wrapper/requirements.txt")
    
    run_cmd(f"git clone https://github.com/chflame163/ComfyUI_LayerStyle.git {COMFY_DIR}/custom_nodes/ComfyUI_LayerStyle 2>/dev/null || true")
    run_cmd(f"pip install -q -r {COMFY_DIR}/custom_nodes/ComfyUI_LayerStyle/requirements.txt")
    
    # 🚨 ใช้ HF CLI ดึงคีย์อัตโนมัติ
    print("🧠 [CatVTON] Downloading Weights via HF API...")
    os.makedirs(f"{COMFY_DIR}/models/checkpoints", exist_ok=True)
    if not os.path.exists(f"{COMFY_DIR}/models/checkpoints/catvton.safetensors"):
        run_cmd(f"huggingface-cli download zhy6769/CatVTON catvton.safetensors --local-dir {COMFY_DIR}/models/checkpoints --local-dir-use-symlinks False")
    
    print("🔨 [4/5] THE HAMMER FIX (CatVTON): Forcing stable dependencies...")
    run_cmd("pip install -q huggingface-hub==0.23.2 diffusers==0.25.0 transformers==4.38.2")

elif TARGET_NODE == "OOTDiffusion":
    run_cmd(f"git clone https://github.com/AuroBit/ComfyUI-OOTDiffusion.git {COMFY_DIR}/custom_nodes/ComfyUI-OOTDiffusion 2>/dev/null || true")
    run_cmd(f"pip install -q -r {COMFY_DIR}/custom_nodes/ComfyUI-OOTDiffusion/requirements.txt")
    
    # 🚨 ใช้ HF CLI 
    os.makedirs(f"{COMFY_DIR}/custom_nodes/ComfyUI-OOTDiffusion/checkpoints", exist_ok=True)
    if not os.path.exists(f"{COMFY_DIR}/custom_nodes/ComfyUI-OOTDiffusion/checkpoints/human_parsing.bin"):
        print("🧠 [OOTDiffusion] Downloading parsing model via HF API...")
        run_cmd(f"huggingface-cli download levihsu/OOTDiffusion checkpoints/human_parsing.bin --local-dir {COMFY_DIR}/custom_nodes/ComfyUI-OOTDiffusion/ --local-dir-use-symlinks False")
        
    print("🔨 [4/5] THE HAMMER FIX (OOTDiffusion): Forcing stable dependencies...")
    run_cmd("pip install -q huggingface-hub==0.23.2 diffusers==0.25.0 transformers==4.38.2")

elif TARGET_NODE == "FitDiT":
    run_cmd(f"git clone https://github.com/BoyuanJiang/FitDiT-ComfyUI.git {COMFY_DIR}/custom_nodes/FitDiT-ComfyUI 2>/dev/null || true")
    run_cmd(f"pip install -q -r {COMFY_DIR}/custom_nodes/FitDiT-ComfyUI/requirements.txt")
    print("🔨 [4/5] THE HAMMER FIX (FitDiT): Forcing stable dependencies...")
    run_cmd("pip install -q huggingface-hub==0.23.2 diffusers==0.25.0 transformers==4.38.2")

print("🌐 [5/5] Launching Cloudflare Tunnel...")
os.chdir(BASE_DIR)
run_cmd("wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb")
run_cmd("dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1")

def iframe_thread(port):
    while True:
        time.sleep(0.5)
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        if sock.connect_ex(('127.0.0.1', port)) == 0: break
        sock.close()

    p = subprocess.Popen(["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{port}"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    for line in p.stderr:
        l = line.decode()
        if "trycloudflare.com " in l:
            print("\n" + "🌟"*30)
            print(f"🚀 COMFYUI READY! คลิกลิงก์ด้านล่างเพื่อเข้าเว็บ 👉:")
            print(f"🔗 URL: {l[l.find('http'):].strip()}")
            print("🌟"*30 + "\n")

threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()

print("🔥 Starting ComfyUI Server on Colab GPU...")
os.chdir(COMFY_DIR)
# 🎯 รันด้วย Granular Precision 
!python main.py --dont-print-server --highvram --fp16-unet --fp16-vae